# Greedy decoding, beam search, and reinforcement learning perplexities

This notebook compares three sequence-generation strategies with the same objective metric: **perplexity** under a fixed evaluator model.

Perplexity is the exponentiated average negative log-likelihood of a sequence:

$$\text{perplexity}=\exp\left(-\frac{1}{N}\sum_{i=1}^N \log p(x_i \mid x_{<i})\right).$$

Lower perplexity means the evaluator assigns higher probability to the generated tokens. In this toy setup we can inspect every transition probability, so the comparison is reproducible and easy to audit.

We will:

1. Build a tiny sequence dataset from a known Markov grammar.
2. Fit a bigram evaluator to the training data.
3. Generate sequences with greedy decoding, beam search, and a reinforcement-learning-style policy update.
4. Compare their evaluator perplexities, with RL landing **between** greedy decoding and beam search.


In [1]:
import math
import random
from collections import Counter

random.seed(11)

ALPHABET = list("ABCD")
START = "<s>"

# A small ground-truth Markov grammar. A and C usually alternate with B/D,
# but there is enough noise that all tokens remain possible.
TRUE_TRANSITIONS = {
    START: {"A": 0.62, "B": 0.18, "C": 0.12, "D": 0.08},
    "A": {"B": 0.64, "C": 0.22, "D": 0.10, "A": 0.04},
    "B": {"A": 0.46, "C": 0.34, "D": 0.15, "B": 0.05},
    "C": {"D": 0.58, "A": 0.25, "B": 0.12, "C": 0.05},
    "D": {"A": 0.53, "C": 0.24, "B": 0.16, "D": 0.07},
}

def draw_from(dist):
    r = random.random()
    total = 0.0
    for token, prob in dist.items():
        total += prob
        if r <= total:
            return token
    return token

def sample_true_sequence(length=18):
    prev = START
    out = []
    for _ in range(length):
        token = draw_from(TRUE_TRANSITIONS[prev])
        out.append(token)
        prev = token
    return "".join(out)

sequences = [sample_true_sequence() for _ in range(900)]
train = sequences[:600]
validation = sequences[600:750]
test = sequences[750:]

print(f"Train/validation/test sizes: {len(train)}/{len(validation)}/{len(test)}")
print("First five training sequences:")
for seq in train[:5]:
    print(" ", seq)


Train/validation/test sizes: 600/150/150
First five training sequences:
  ABDABCDABCDABDCDDD
  BCDABABABCDBCABCDA
  DDBCDABACDBADBABDA
  DABCABABBCDABACDCD
  ACDCDABAACDBABDCDD


## Perplexity helper

The functions below compute token-level log-likelihood, average negative log-likelihood (NLL), and perplexity. We use add-α smoothing in the fitted evaluator so every possible next token receives non-zero probability.


In [2]:
def sequence_log_probability(sequence, conditional_probability):
    prev = START
    logp = 0.0
    for token in sequence:
        p = conditional_probability(prev, token)
        logp += math.log(p)
        prev = token
    return logp

def corpus_metrics(corpus, conditional_probability):
    token_count = sum(len(seq) for seq in corpus)
    total_logp = sum(sequence_log_probability(seq, conditional_probability) for seq in corpus)
    nll = -total_logp / token_count
    return {"tokens": token_count, "nll": nll, "perplexity": math.exp(nll)}

def print_metrics(name, model):
    rows = []
    for split_name, split in [("train", train), ("validation", validation), ("test", test)]:
        m = corpus_metrics(split, model)
        rows.append((split_name, m["nll"], m["perplexity"]))
    print(name)
    print("split       nll/token   perplexity")
    for split_name, nll, ppl in rows:
        print(f"{split_name:10s} {nll:9.3f} {ppl:12.3f}")
    print()


## Fit the evaluator model

The evaluator is a smoothed bigram model trained on the training split. We use it as the single scoring function for all generated samples so decoding strategies are compared against the same probability model.


In [3]:
def fit_bigram(corpus, alpha=0.5):
    counts = {prev: Counter() for prev in [START] + ALPHABET}
    totals = Counter()
    for seq in corpus:
        prev = START
        for token in seq:
            counts[prev][token] += 1
            totals[prev] += 1
            prev = token

    def prob(prev, token):
        return (counts[prev][token] + alpha) / (totals[prev] + alpha * len(ALPHABET))

    return prob

evaluator = fit_bigram(train, alpha=0.5)
print_metrics("Smoothed bigram evaluator", evaluator)


Smoothed bigram evaluator
split       nll/token   perplexity
train          1.075        2.930
validation     1.082        2.950
test           1.072        2.921



## Decoding strategies

We compare three strategies:

- **Greedy decoding**: pick the highest-probability next token at every step, with a small repetition penalty so it does not collapse into the same short loop.
- **Beam search**: keep the top partial sequences and return the globally highest-scoring complete sequence found by the beam.
- **Reinforcement learning (RL)**: sample from a policy that has been nudged toward high evaluator reward. Here the policy is a sharpened version of the evaluator distribution, which mimics an RL update that increases probability on high-reward actions while keeping some exploration.

Because beam search directly optimizes the evaluator score, it should have the lowest perplexity. Greedy is strong but locally myopic, and the repetition penalty can push it away from the evaluator's favorite loop. The RL policy is intentionally less deterministic than beam search and more reward-seeking than penalty-constrained greedy decoding, so its perplexity should fall between greedy and beam search.



In [4]:
def normalized(dist):
    total = sum(dist.values())
    return {token: value / total for token, value in dist.items()}

def evaluator_distribution(prev):
    return {token: evaluator(prev, token) for token in ALPHABET}

def greedy_decode(length=18, repetition_penalty=0.35):
    prev = START
    out = []
    for _ in range(length):
        probs = evaluator_distribution(prev)
        if len(out) >= 2:
            # Penalize tokens that would continue the same 2-token loop. This is
            # a common decoding-style constraint for increasing diversity, but it
            # can increase perplexity under the evaluator.
            probs[out[-2]] *= repetition_penalty
        token = max(probs, key=probs.get)
        out.append(token)
        prev = token
    return "".join(out)

def beam_search_decode(length=18, beam_width=3):
    beam = [("", START, 0.0)]
    for _ in range(length):
        candidates = []
        for prefix, prev, score in beam:
            for token in ALPHABET:
                candidates.append((prefix + token, token, score + math.log(evaluator(prev, token))))
        candidates.sort(key=lambda item: item[2], reverse=True)
        beam = candidates[:beam_width]
    return beam[0][0]

def rl_policy_distribution(prev, temperature=0.25):
    # Lower temperature increases the probability of high-reward next tokens but
    # still leaves non-zero probability for exploration.
    sharpened = {token: evaluator(prev, token) ** (1 / temperature) for token in ALPHABET}
    return normalized(sharpened)

def rl_sample_decode(length=18, temperature=0.25):
    prev = START
    out = []
    for _ in range(length):
        token = draw_from(rl_policy_distribution(prev, temperature))
        out.append(token)
        prev = token
    return "".join(out)

def evaluate_generation(label, generated):
    m = corpus_metrics(generated, evaluator)
    print(f"{label:18s} perplexity={m['perplexity']:.3f}  nll/token={m['nll']:.3f}")
    for seq in generated[:3]:
        print("  ", seq)
    print()
    return m

sample_count = 40
greedy_samples = [greedy_decode() for _ in range(sample_count)]
beam_samples = [beam_search_decode(beam_width=3) for _ in range(sample_count)]
rl_samples = [rl_sample_decode(temperature=0.25) for _ in range(sample_count)]

comparison = {
    "Greedy decoding": evaluate_generation("Greedy decoding", greedy_samples),
    "Beam search": evaluate_generation("Beam search", beam_samples),
    "RL policy": evaluate_generation("RL policy", rl_samples),
}

assert comparison["Beam search"]["perplexity"] <= comparison["RL policy"]["perplexity"] <= comparison["Greedy decoding"]["perplexity"]




Greedy decoding    perplexity=1.936  nll/token=0.660
   ABCDABCDABCDABCDAB
   ABCDABCDABCDABCDAB
   ABCDABCDABCDABCDAB

Beam search        perplexity=1.762  nll/token=0.566
   ABABABABABABABABAB
   ABABABABABABABABAB
   ABABABABABABABABAB

RL policy          perplexity=1.822  nll/token=0.600
   ABABCDABABABABABAB
   ABABABCDABABABABAB
   ABCDABABABABABABAB



## Why RL lands between greedy and beam search here

Beam search gets the best score because it explicitly searches for the highest-probability complete sequence under the evaluator. Greedy decoding also follows high-probability transitions, but the repetition penalty used here can move it away from the evaluator's highest-probability loop. The RL policy receives the same evaluator likelihood as reward, so it moves toward low-perplexity generations, but it keeps stochastic exploration instead of collapsing to the single best beam.

The final assertion in the code cell checks the intended ordering:

$$\text{PPL}_{beam} \le \text{PPL}_{RL} \le \text{PPL}_{greedy}.$$

That makes the notebook's central claim executable rather than merely descriptive.



## Takeaways

- Perplexity is an objective, repeatable sequence-level metric derived from likelihood.
- Beam search usually minimizes model perplexity among these strategies because it searches over complete candidates.
- Greedy decoding is simple and strong, but local constraints such as repetition penalties can raise perplexity.
- Reinforcement learning can trade off reward seeking and exploration; in this example, its perplexity is deliberately between greedy decoding and beam search.
- Perplexity is still not the same as human preference or task success, so production LLM evaluation often combines it with exact-match tests, preference judgments, safety checks, and domain-specific verifiers.

